# Spheroid symmetry verification for block-DDA_Py

Verify that DDA results for a spheroid (GRE with `beta=0`, `ab_ratio=1`) satisfy
the analytical constraint relations derived from the particle's axial symmetry.

Euler angle mapping: $\alpha$ = azimuthal angle $\phi$, $\beta$ = polar angle $\theta$, $\gamma$ = irrelevant for spheroid.

In [ ]:
import numpy as np
from shape_model.gaussian_ellipsoid import gaussian_ellipsoid_shape_model
from analytical_scattering_theories.homogeneous_sphere import mie_compute_q_and_s
from bl_dda.scatterer import Target, IncidentField, DiscreteDipoles

In [ ]:
# ── Spheroid parameters ──
# Oblate spheroid: a = b, c < b  (bc_ratio = b/c > 1)
r_v_base = 0.2          # volume-equivalent radius [um]
bc_ratio = 2.0          # b/c ratio (oblate)
ab_ratio = 1.0          # a/b = 1 (axial symmetry required)
beta_gre = 0.0          # no surface roughness (smooth ellipsoid)

# Wavelength and refractive index
wl_0  = 0.834           # vacuum wavelength [um]
m_m   = 1.329           # medium refractive index (real)
m_p_x = 1.5 + 0.01j
m_p_y = 1.5 + 0.01j
m_p_z = 1.5 + 0.01j
m_p_xyz = np.array([m_p_x, m_p_y, m_p_z], dtype=np.complex128)

# Orientations: fixed beta (polar angle), sweep alpha (azimuthal), gamma=0
beta_deg  = 45.0                                  # fixed polar angle
alpha_deg = np.arange(0, 360, 30, dtype=float)    # 12 azimuthal angles
N_alpha   = len(alpha_deg)

euler_angles = np.column_stack([
    np.radians(alpha_deg),
    np.full(N_alpha, np.radians(beta_deg)),
    np.zeros(N_alpha),                             # gamma = 0
])
print(f"Spheroid: r_v_base={r_v_base}, bc_ratio={bc_ratio}, beta={beta_deg}°")
print(f"Orientations: {N_alpha} alpha values from 0° to 330°")

In [ ]:
# ── Generate spheroid lattice and Target ──
rng = np.random.default_rng(1234)
gre = gaussian_ellipsoid_shape_model(r_v_base, bc_ratio, ab_ratio, beta_gre, wl_0, m_p_xyz)
r_pts, _ = gre.compute_r_points_on_GRE(rng)
_, lattice_n, grid = gre.create_cuboid_lattice_that_encloses_GRE_shape(r_pts)
dist = gre.find_nearest_distance_from_the_GRE_surf(grid, r_pts)
is_in = gre.extract_lattice_address_in_GRE_volume(gre.lattice_lf, gre.distance_factor, lattice_n, dist)

target = Target(gre.name, lattice_n, gre.lattice_lf, grid, is_in, m_p_xyz, r_v_base)
print(f"lattice_n = {lattice_n}, N_occ = {target.num_element_occupy}")
print(f"ve_radius = {target.ve_radius:.6f} um  (r_v_base = {r_v_base})")

In [ ]:
# ── DDA solve for all 12 orientations ──
inc = IncidentField(wl_0, m_m, euler_angles)
dd  = DiscreteDipoles(target, inc)
dd.set_interaction_matrix()
dd.solve_matrix_equation()

In [ ]:
# ── Compute observables ──
# S_s ≡ S_fw_PCAS_theta,  S_p ≡ S_fw_PCAS_phi
S_s, S_p = dd.compute_PCAS_observable_S_fw()
C_abs = dd.compute_C_abs()
C_ext = dd.compute_C_ext()

alpha_rad = np.radians(alpha_deg)
print("S_s (S_fw_theta) for each alpha:")
for i, a in enumerate(alpha_deg):
    print(f"  alpha={a:5.0f}°:  S_s = {S_s[i]:.6f},  S_p = {S_p[i]:.6f}")

## Verification A: Trace invariance
$S_s(\alpha) + S_p(\alpha) = S_{11}(\theta,0) + S_{22}(\theta,0) = \text{const}$ for all $\alpha$.

In [ ]:
# Verification A: Trace invariance
trace = S_s + S_p                     # should be constant for all alpha
trace_ref = trace[0]                  # reference value (alpha=0)
trace_dev = np.abs(trace - trace_ref) / np.abs(trace_ref)

print("Trace  S_s + S_p:")
for i, a in enumerate(alpha_deg):
    print(f"  alpha={a:5.0f}°:  trace = {trace[i]:.8f}   rel_dev = {trace_dev[i]:.2e}")
print(f"\n  max relative deviation = {trace_dev.max():.2e}")
assert trace_dev.max() < 1e-3, f"FAIL: trace deviation {trace_dev.max():.2e} > 1e-3"
print("  ✓ PASS")

## Verification B: Modulus preservation
$|S_s(\alpha) - S_p(\alpha)| = |S_{11}(\theta,0) - S_{22}(\theta,0)| = \text{const}$ for all $\alpha$.

In [ ]:
# Verification B: Modulus preservation
diff_mod = np.abs(S_s - S_p)         # should be constant for all alpha
diff_ref = diff_mod[0]
mod_dev  = np.abs(diff_mod - diff_ref) / diff_ref

print("|S_s - S_p|:")
for i, a in enumerate(alpha_deg):
    print(f"  alpha={a:5.0f}°:  |S_s - S_p| = {diff_mod[i]:.8f}   rel_dev = {mod_dev[i]:.2e}")
print(f"\n  max relative deviation = {mod_dev.max():.2e}")
assert mod_dev.max() < 1e-3, f"FAIL: modulus deviation {mod_dev.max():.2e} > 1e-3"
print("  ✓ PASS")

## Verification C: Cross-symmetry
$S_s(\theta, \alpha + \pi/2) = S_p(\theta, \alpha)$ for all $\alpha$.

In [ ]:
# Verification C: Cross-symmetry  S_s(alpha + 90°) = S_p(alpha)
# alpha = [0, 30, 60, 90, 120, ...] with spacing 30°
# alpha + 90° shifts by 3 indices (90/30 = 3)
shift = 3   # 90° / 30° = 3 positions
S_s_shifted = np.roll(S_s, -shift)   # S_s at alpha + 90°

cross_dev = np.abs(S_s_shifted - S_p) / np.abs(S_p)

print("Cross-symmetry  S_s(alpha+90°) vs S_p(alpha):")
for i, a in enumerate(alpha_deg):
    a_shifted = (a + 90) % 360
    print(f"  S_s({a_shifted:5.0f}°) = {S_s_shifted[i]:.6f}   "
          f"S_p({a:5.0f}°) = {S_p[i]:.6f}   rel_dev = {cross_dev[i]:.2e}")
print(f"\n  max relative deviation = {cross_dev.max():.2e}")
assert cross_dev.max() < 1e-3, f"FAIL: cross-symmetry deviation {cross_dev.max():.2e} > 1e-3"
print("  ✓ PASS")

## Verification D: Analytical formula
Using $S_{11}(\theta,0) = S_s(\alpha\!=\!0)$ and $S_{22}(\theta,0) = S_p(\alpha\!=\!0)$ from the DDA at $\alpha=0$, predict $S_s$ and $S_p$ at all other $\alpha$ values analytically:\n\n$$S_s(\alpha) = \\frac{S_{11} + S_{22}}{2} + \\frac{S_{11} - S_{22}}{2}\\,e^{i2\\alpha}$$\n$$S_p(\\alpha) = \\frac{S_{11} + S_{22}}{2} - \\frac{S_{11} - S_{22}}{2}\\,e^{i2\\alpha}$$

In [ ]:
# Verification D: Analytical formula using alpha=0 reference
S11_ref = S_s[0]   # S_s at alpha=0 = S_11(theta, 0)
S22_ref = S_p[0]   # S_p at alpha=0 = S_22(theta, 0)

A = (S11_ref + S22_ref) / 2
B = (S11_ref - S22_ref) / 2

S_s_pred = A + B * np.exp(2j * alpha_rad)
S_p_pred = A - B * np.exp(2j * alpha_rad)

dev_s = np.abs(S_s - S_s_pred) / np.abs(S_s)
dev_p = np.abs(S_p - S_p_pred) / np.abs(S_p)

print("Analytical formula vs DDA:")
print(f"  {'alpha':>7s}  {'dev_Ss':>10s}  {'dev_Sp':>10s}")
for i, a in enumerate(alpha_deg):
    print(f"  {a:5.0f}°   {dev_s[i]:10.2e}  {dev_p[i]:10.2e}")
print(f"\n  max dev(S_s) = {dev_s.max():.2e}")
print(f"  max dev(S_p) = {dev_p.max():.2e}")
assert dev_s.max() < 1e-3, f"FAIL: S_s formula deviation {dev_s.max():.2e} > 1e-3"
assert dev_p.max() < 1e-3, f"FAIL: S_p formula deviation {dev_p.max():.2e} > 1e-3"
print("  ✓ PASS")

## Verification E: Phi-averaged observables
The phi-averaged forward scattering amplitudes should equal $(S_{11}(\theta,0) + S_{22}(\theta,0))/2$.\n\nAlso verify that the phi-averaged $C_\mathrm{ext}$ and $C_\mathrm{abs}$ are consistent with the analytical phi-average of the optical theorem.

In [ ]:
# Verification E: Phi-averaged observables
S_avg_analytical = (S11_ref + S22_ref) / 2   # analytical phi-average
S_s_mean = S_s.mean()
S_p_mean = S_p.mean()

print("Phi-averaged S_s and S_p (numerical mean over 12 alpha values):")
print(f"  <S_s>_phi = {S_s_mean:.8f}")
print(f"  <S_p>_phi = {S_p_mean:.8f}")
print(f"  Analytical = {S_avg_analytical:.8f}")
print(f"  rel_dev(<S_s>) = {abs(S_s_mean - S_avg_analytical)/abs(S_avg_analytical):.2e}")
print(f"  rel_dev(<S_p>) = {abs(S_p_mean - S_avg_analytical)/abs(S_avg_analytical):.2e}")

# Phi-averaged C_ext: numerical vs alpha=0 computation
C_ext_mean = C_ext.mean()
C_ext_alpha0 = C_ext[0]
print(f"\n  <C_ext>_phi (numerical) = {C_ext_mean:.6e}")
print(f"  C_ext(alpha=0)          = {C_ext_alpha0:.6e}")

# Phi-averaged C_abs
C_abs_mean = C_abs.mean()
C_abs_alpha0 = C_abs[0]
print(f"  <C_abs>_phi (numerical) = {C_abs_mean:.6e}")
print(f"  C_abs(alpha=0)          = {C_abs_alpha0:.6e}")

## Verification F: Sphere limit ($b/c = 1$)\n$S_s = S_p$ for all $\\alpha$, and both should match the Mie solution.

In [ ]:
# Verification F: Sphere limit  (bc_ratio = 1, ab_ratio = 1)
bc_ratio_sph = 1.0
rng_sph = np.random.default_rng(1234)
gre_sph = gaussian_ellipsoid_shape_model(r_v_base, bc_ratio_sph, ab_ratio, beta_gre, wl_0, m_p_xyz)
r_pts_sph, _ = gre_sph.compute_r_points_on_GRE(rng_sph)
_, n_sph, grid_sph = gre_sph.create_cuboid_lattice_that_encloses_GRE_shape(r_pts_sph)
dist_sph = gre_sph.find_nearest_distance_from_the_GRE_surf(grid_sph, r_pts_sph)
is_in_sph = gre_sph.extract_lattice_address_in_GRE_volume(
    gre_sph.lattice_lf, gre_sph.distance_factor, n_sph, dist_sph)

target_sph = Target(gre_sph.name, n_sph, gre_sph.lattice_lf, grid_sph, is_in_sph, m_p_xyz, r_v_base)
inc_sph = IncidentField(wl_0, m_m, euler_angles)
dd_sph  = DiscreteDipoles(target_sph, inc_sph)
dd_sph.set_interaction_matrix()
dd_sph.solve_matrix_equation()

S_s_sph, S_p_sph = dd_sph.compute_PCAS_observable_S_fw()

# S_s should equal S_p for a sphere (no orientation dependence)
depol = np.abs(S_s_sph - S_p_sph) / np.abs(S_s_sph)
print("Sphere limit: S_s vs S_p")
for i, a in enumerate(alpha_deg):
    print(f"  alpha={a:5.0f}°:  |S_s - S_p|/|S_s| = {depol[i]:.2e}")
print(f"  max depolarisation = {depol.max():.2e}")

# Compare with Mie
m_p_avg = complex(np.mean(m_p_xyz))
_, _, _, S_fw_mie, _ = mie_compute_q_and_s(wl_0, m_m, target_sph.ve_radius, m_p_avg, nang=3)
mie_dev_s = np.abs(S_s_sph - S_fw_mie) / np.abs(S_fw_mie)
mie_dev_p = np.abs(S_p_sph - S_fw_mie) / np.abs(S_fw_mie)
print(f"\nMie comparison (S_fw_mie = {S_fw_mie:.6f}):")
print(f"  max |S_s - Mie|/|Mie| = {mie_dev_s.max():.2e}")
print(f"  max |S_p - Mie|/|Mie| = {mie_dev_p.max():.2e}")
assert mie_dev_s.max() < 1e-2, f"FAIL: Mie deviation {mie_dev_s.max():.2e} > 1e-2"
print("  ✓ PASS")

## Verification G: Spheroid mode (Phase 2)
Test the labor-saving mode: sample only $\beta$ (alpha=0, gamma=0), then compute phi-averaged S analytically.\nCompare with brute-force phi-averaging from Verification A–D.

In [ ]:
# Verification G: Spheroid mode — single alpha=0 solve + analytical phi-average
# Use the same beta=45° as before, but only alpha=0
euler_single = np.array([[0.0, np.radians(45.0), 0.0]])   # shape (1, 3)

inc_single = IncidentField(wl_0, m_m, euler_single)
dd_single  = DiscreteDipoles(target, inc_single)
dd_single.set_interaction_matrix()
dd_single.solve_matrix_equation()

S_s_0, S_p_0 = dd_single.compute_PCAS_observable_S_fw()
S_phi_avg = dd_single.compute_phi_averaged_PCAS_S_fw()
C_ext_0 = dd_single.compute_C_ext()
C_abs_0 = dd_single.compute_C_abs()

# Compare with brute-force phi-average from the 12-alpha calculation
print("Spheroid mode: single alpha=0 solve + analytical phi-average")
print(f"  S_s(alpha=0)   = {S_s_0[0]:.8f}")
print(f"  S_p(alpha=0)   = {S_p_0[0]:.8f}")
print(f"  <S>_phi (anal) = {S_phi_avg[0]:.8f}")
print(f"  <S_s>_phi (brute-force, 12 alpha) = {S_s.mean():.8f}")
print(f"  <S_p>_phi (brute-force, 12 alpha) = {S_p.mean():.8f}")

dev_S = abs(S_phi_avg[0] - S_s.mean()) / abs(S_s.mean())
dev_Cext = abs(C_ext_0[0] - C_ext.mean()) / abs(C_ext.mean())
dev_Cabs = abs(C_abs_0[0] - C_abs.mean()) / abs(C_abs.mean())

print(f"\n  rel_dev(<S>)    = {dev_S:.2e}")
print(f"  rel_dev(<C_ext>) = {dev_Cext:.2e}")
print(f"  rel_dev(<C_abs>) = {dev_Cabs:.2e}")
assert dev_S < 1e-10, f"FAIL: S phi-average deviation {dev_S:.2e}"
print("  ✓ PASS: spheroid mode reproduces brute-force phi-average")